# 09 — Optimizers and Training Dynamics

Demonstrate a small training loop, compare optimizer settings, and define checkpointing and reproducibility practices.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
X = torch.randn(256, 10)
y = (X[:, 0] + 0.5 * X[:, 1] > 0).float().unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Toy example only; not trained on LAV-DF.")

In [ ]:
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
    print(f"epoch={epoch+1} loss={running_loss/len(loader.dataset):.4f}")

## Optimizer notes
SGD updates parameters using gradients and a learning rate; momentum smooths updates. Adam adapts per-parameter step sizes and is a reasonable baseline. Tune learning rate on development data, save model/optimizer/epoch/seed, and reload the best checkpoint rather than the final epoch by default.

In [ ]:
from pathlib import Path
CHECKPOINT_DIR = Path("../../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory:", CHECKPOINT_DIR.resolve())
# Example: torch.save({"model": model.state_dict(),
#                      "optimizer": optimizer.state_dict(),
#                      "epoch": epoch, "seed": 42},
#                     CHECKPOINT_DIR / "toy_checkpoint.pt")